#### Contabilidad mensual

Esta notebook tiene como objetivo leer archivos .xls que se le envian todos los meses que corresponden a compras y ventas

In [2]:
### Libreris
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
print ("Liberias ok")

Liberias ok


In [ ]:
#archivo
archivo = Path("../data/raw/contabilidad/CONTABILIDAD 2026-01_ISLA.xlsx")
#Vemos cuales son los titulos de las hojas
excel = pd.ExcelFile(archivo)
excel.sheet_names


['COMPRAS', 'VENTAS']

In [36]:
#Se carga cada archivo por separado
df_compras = pd.read_excel(archivo, sheet_name="COMPRAS")
df_ventas = pd.read_excel(archivo, sheet_name="VENTAS")

In [8]:
#se prueba que sse cargue correctamente
df_compras.head()

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17
0,Razon Social:,ISLA ECOLOGICA S.A.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,CUIT,30-71543101-3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,MES,Enero,2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Fecha,Tipo Compr.,Punto de venta,Nº Compr.,Razon Social,Cuit,Condición,Concepto,Forma de Pago,Neto Gravado,"Iva % 10,5",Iva % 21,Iva 27%,IVA 3%,Perc. IIBB,Perc. Munic.,No Gravado y Exento,Total


In [9]:
#se prueba que sse cargue correctamente
df_ventas.head()

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15
0,Razon Social:,ISLA ECOLOGICA S.A.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,CUIT,30-71543101-3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,MES,Enero,2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Fecha,Tipo Compr.,Punto de venta,Nro. Compr.,Razón Social,Cuit/DNI,Neto Gravado 1*,"IVA % 10,5",IVA % 21,Iva 27%,IVA 3%,Perc. IIBB,Perc. Munic.,No Gravado y Exento,Total,Forma de Cobro


En ambos casos se ve que las primeras filas no hay que considerarlas y que la 4ta corresponde a los titulos

In [37]:
#Los volvemos a abrir sacando las primeras columnas
df_compras = pd.read_excel(archivo, sheet_name="COMPRAS", header=5)
df_ventas = pd.read_excel(archivo, sheet_name="VENTAS", header=5)

In [ ]:
df_compras.head()

In [ ]:
df_ventas.head()

In [ ]:
df_ventas.tail()

Esto era un solo archivo ahora lo aplicamos a toda la carpeta

In [77]:
carpeta = Path("../data/raw/contabilidad")
archivos = sorted(carpeta.glob("*.xlsx"))
compras = []
ventas = []

for archivo in archivos:

    # Compras
    df_c = pd.read_excel(archivo, sheet_name="COMPRAS", header=5)
    # Ventas
    df_v = pd.read_excel(archivo, sheet_name="VENTAS", header=5)

    # Trazabilidad
    df_c["archivo_origen"] = archivo.name
    df_v["archivo_origen"] = archivo.name

    compras.append(df_c)
    ventas.append(df_v)

df_compras = pd.concat(compras, ignore_index=True)
df_ventas = pd.concat(ventas, ignore_index=True)

#Eliminamos columnas sin nombrar que pudan estar en algunos archivos
df_compras = df_compras.loc[:, ~df_compras.columns.str.contains("^Unnamed")]
df_ventas = df_ventas.loc[:, ~df_ventas.columns.str.contains("^Unnamed")]

Se hace un pequeño EDA para explorar la informacion que tenemos en cada uno de los archivos
para reponder a preguntas como:
- columnas vacías,
- tipos de datos incorrectos,
- formatos de fecha distintos,
- columnas innecesarias,
- diferencias entre meses.

In [60]:
df_compras.info()

<class 'pandas.DataFrame'>
RangeIndex: 394 entries, 0 to 393
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Fecha                394 non-null    object 
 1   Tipo Compr.          394 non-null    str    
 2   Punto de venta       394 non-null    int64  
 3   Nº Compr.            394 non-null    float64
 4   Razon Social         394 non-null    str    
 5   Cuit                 394 non-null    object 
 6   Condición            394 non-null    str    
 7   Concepto             394 non-null    str    
 8   Forma de Pago        394 non-null    str    
 9   Neto Gravado         394 non-null    float64
 10  Iva % 10,5           8 non-null      float64
 11  Iva % 21             394 non-null    float64
 12  Iva 27%              2 non-null      float64
 13  IVA 3%               24 non-null     float64
 14  Perc. IIBB           10 non-null     float64
 15  Perc. Munic.         24 non-null     float64
 16  N

In [61]:
df_compras.columns

Index(['Fecha', 'Tipo Compr.', 'Punto de venta', 'Nº Compr.', 'Razon Social',
       'Cuit', 'Condición', 'Concepto', 'Forma de Pago', 'Neto Gravado',
       'Iva % 10,5', 'Iva % 21 ', 'Iva 27%', 'IVA 3%', 'Perc. IIBB',
       'Perc. Munic.', 'No Gravado y Exento', 'Total', 'archivo_origen'],
      dtype='str')

In [54]:
df_compras.dtypes

Fecha                   object
Tipo Compr.                str
Punto de venta         float64
Nº Compr.              float64
Razon Social               str
Cuit                    object
Condición                  str
Concepto                   str
Forma de Pago              str
Neto Gravado           float64
Iva % 10,5             float64
Iva % 21               float64
Iva 27%                float64
IVA 3%                 float64
Perc. IIBB             float64
Perc. Munic.           float64
No Gravado y Exento    float64
Total                  float64
archivo_origen             str
dtype: object

In [62]:
df_compras.isna().sum()
# df_compras[df_compras["Fecha"].isna()]['archivo_origen']

Fecha                    0
Tipo Compr.              0
Punto de venta           0
Nº Compr.                0
Razon Social             0
Cuit                     0
Condición                0
Concepto                 0
Forma de Pago            0
Neto Gravado             0
Iva % 10,5             386
Iva % 21                 0
Iva 27%                392
IVA 3%                 370
Perc. IIBB             384
Perc. Munic.           370
No Gravado y Exento    225
Total                    0
archivo_origen           0
dtype: int64

In [63]:
# ventas
df_ventas.info()

<class 'pandas.DataFrame'>
RangeIndex: 464 entries, 0 to 463
Data columns (total 21 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Fecha                464 non-null    str    
 1   Tipo Compr.          464 non-null    str    
 2   Punto de venta       464 non-null    int64  
 3   Nro. Compr.          399 non-null    float64
 4   Razón Social         399 non-null    str    
 5   Cuit/DNI             38 non-null     float64
 6   Neto Gravado 1*      464 non-null    float64
 7   IVA % 10,5           0 non-null      float64
 8   IVA % 21             399 non-null    float64
 9   Iva 27%              0 non-null      float64
 10  IVA 3%               0 non-null      float64
 11  Perc. IIBB           0 non-null      float64
 12  Perc. Munic.         0 non-null      float64
 13  No Gravado y Exento  0 non-null      float64
 14  Total                464 non-null    float64
 15  Forma de Cobro       464 non-null    str    
 16  a

In [64]:
df_ventas.columns

Index(['Fecha', 'Tipo Compr.', 'Punto de venta', 'Nro. Compr.', 'Razón Social',
       'Cuit/DNI', 'Neto Gravado 1*', 'IVA % 10,5', 'IVA % 21', 'Iva 27%',
       'IVA 3%', 'Perc. IIBB', 'Perc. Munic.', 'No Gravado y Exento', 'Total',
       'Forma de Cobro', 'archivo_origen', 'Nº Compr.', 'Razon Social',
       'Iva % 10,5', 'Iva % 21 '],
      dtype='str')

In [65]:
df_ventas.dtypes

Fecha                      str
Tipo Compr.                str
Punto de venta           int64
Nro. Compr.            float64
Razón Social               str
Cuit/DNI               float64
Neto Gravado 1*        float64
IVA % 10,5             float64
IVA % 21               float64
Iva 27%                float64
IVA 3%                 float64
Perc. IIBB             float64
Perc. Munic.           float64
No Gravado y Exento    float64
Total                  float64
Forma de Cobro             str
archivo_origen             str
Nº Compr.              float64
Razon Social               str
Iva % 10,5             float64
Iva % 21               float64
dtype: object

In [78]:
df_ventas.isna().sum()
# df_ventas.loc[
#     df_ventas["IVA % 21"].isna(),
#     ["archivo_origen", "Fecha", "IVA % 21"]
# ]

Fecha                    0
Tipo Compr.              0
Punto de venta           0
Nro. Compr.             65
Razón Social             0
Cuit/DNI               426
Neto Gravado 1*          0
IVA % 10,5             464
IVA % 21                 0
Iva 27%                464
IVA 3%                 464
Perc. IIBB             464
Perc. Munic.           464
No Gravado y Exento    464
Total                    0
Forma de Cobro           0
archivo_origen           0
Nº Compr.              399
Iva % 10,5             464
dtype: int64

In [38]:
def normalizar_columnas(df):
    return df.rename(columns={
        "Fecha": "fecha",
        "Tipo Compr.": "tipo_comprobante",
        "Punto de venta": "punto_venta",
        "Nº Compr.": "numero_comprobante",
        "Nro. Compr.": "numero_comprobante",
        "Razon Social": "razon_social",
        "Razón Social": "razon_social",
        "Cuit": "cuit",
        "Cuit/DNI": "cuit",
        "Condición": "condicion",
        "Concepto": "concepto",
        "Forma de Pago": "forma_pago",
        "Forma de Cobro": "forma_cobro",
        "Neto Gravado": "neto_gravado",
        "Neto Gravado 1*": "neto_gravado",
        "Iva % 10,5": "iva_10_5",
        "IVA % 10,5": "iva_10_5",
        "Iva % 21 ": "iva_21",
        "IVA % 21": "iva_21",
        "Iva 27%": "iva_27",
        "IVA 3%": "iva_3",
        "Perc. IIBB": "percepcion_iibb",
        "Perc. Munic.": "percepcion_municipal",
        "No Gravado y Exento": "no_gravado_exento",
        "Total": "total"
    })
df_compras = normalizar_columnas(df_compras)
df_ventas = normalizar_columnas(df_ventas)

In [43]:
#Formato fecha
# Compras
df_compras["fecha"] = (pd.to_datetime(df_compras["fecha"], dayfirst=True,errors="raise").astype("datetime64[ns]"))

# Ventas
df_ventas["fecha"] = (pd.to_datetime(df_ventas["fecha"], dayfirst=True, errors="raise").astype("datetime64[ns]"))

#S verifican que hayan qudao bien
print(df_compras.dtypes["fecha"])
print(df_ventas.dtypes["fecha"])

print(df_ventas['fecha'].head())
print(df_compras['fecha'].head())

datetime64[ns]
datetime64[ns]
0   2026-01-29
1   2026-01-06
2   2026-01-12
3   2026-01-15
4   2026-01-13
Name: fecha, dtype: datetime64[ns]
0   2026-01-20
1   2026-01-24
2   2026-01-02
3   2026-01-05
4   2026-01-06
Name: fecha, dtype: datetime64[ns]


In [44]:
# Cantidada de registros
print(len(df_compras))
print(len(df_ventas))

76
94


In [45]:
#Duplicados
df_compras.duplicated().sum()
df_ventas.duplicated().sum()

np.int64(0)

In [ ]:
#Valores unicos
df_compras["tipo_comprobante"].value_counts()

In [ ]:
df_compras["forma_pago"].value_counts()

In [ ]:
df_compras["concepto"].value_counts()

In [ ]:
df_ventas["forma_cobro"].value_counts()

In [ ]:
#Distribucion de importes
df_compras["total"].describe()

In [ ]:
df_ventas["total"].describe()

In [ ]:
#Evolucion temproal
compras_mes = (df_compras.groupby(df_compras["fecha"].dt.to_period("M"))["total"].sum())

In [ ]:
df_compras.groupby("razon_social")["total"].sum().sort_values(ascending=False).head(10)

In [ ]:
df_compras["tipo_comprobante"].value_counts()

In [ ]:
df_compras["forma_pago"].value_counts()

In [ ]:
df_compras["iva_21"].sum()

In [ ]:
df_ventas["iva_21"].sum()